# Andere WebArena-Verified Sites mit BrowserGym testen

Nach GitLab Task 44 testen wir hier eine zweite Site. Ziel ist nicht sofort, eine Shopping-/Reddit-Aufgabe zu loesen, sondern kontrolliert zu pruefen:

1. offizielle WebArena-Verified Environment-CLI kann eine Site starten,
2. Config passt zur Site,
3. `agent-input-get` rendert echte URLs,
4. BrowserGym kann die Startseite oeffnen und HAR/Observation erzeugen.

Erst danach kommt ein echter Agent fuer diese Site.

In [8]:
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OFFICIAL_REPO = PROJECT_ROOT / 'external' / 'webarena-verified'
OFFICIAL_REPO

PosixPath('/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified')

## 1. Site auswaehlen

Die aktuelle Experimentalversion testet `shopping`, `shopping_admin`, `reddit`, `gitlab` und optional `wikipedia`. `map` ist wegen des sehr grossen Speicherbedarfs fuer Download und Docker-Volumes bewusst aus dem lokalen Experiment ausgeschlossen. Fuer erste Smoke-Tests sind `shopping`, `shopping_admin` und `reddit` am angenehmsten. `wikipedia` braucht vorher Daten-Setup.

In [9]:
SITE = 'shopping'

SITE_ENV = {
    'shopping': ('__SHOPPING__', 'http://localhost:7770', {'username': 'emma.lopez@gmail.com', 'password': 'Password.123'}),
    'shopping_admin': ('__SHOPPING_ADMIN__', 'http://localhost:7780', {'username': 'admin', 'password': 'admin1234'}),
    'reddit': ('__REDDIT__', 'http://localhost:9999', {'username': 'MarvelsGrantMan136', 'password': 'test1234'}),
    'gitlab': ('__GITLAB__', 'http://localhost:8023', {'username': 'byteblaze', 'password': 'hello1234'}),
    'wikipedia': ('__WIKIPEDIA__', 'http://localhost:8888', None),
   # 'map': ('__MAP__', 'http://localhost:3000', None),
}

SITE_IMAGES = {
    'shopping': 'am1n3e/webarena-verified-shopping',
    'shopping_admin': 'am1n3e/webarena-verified-shopping_admin',
    'reddit': 'am1n3e/webarena-verified-reddit',
    'gitlab': 'am1n3e/webarena-verified-gitlab',
    'wikipedia': 'am1n3e/webarena-verified-wikipedia',
    #'map': 'am1n3e/webarena-verified-map',
}

ALL_SITES = ['shopping', 'shopping_admin', 'reddit', 'gitlab', 'wikipedia']
SITES_WITH_EXTRA_DATA = ['wikipedia']
PREPARE_SITES = ALL_SITES
DATA_DIR = OFFICIAL_REPO / 'downloads'

assert SITE in SITE_ENV
SITE_ENV[SITE]

('__SHOPPING__',
 'http://localhost:7770',
 {'username': 'emma.lopez@gmail.com', 'password': 'Password.123'})

## 2. Images herunterladen und Environment starten

Alle WebArena-Verified-Sites koennen vorab per `docker pull` geladen werden. Danach startest du fuer die eigentliche Probe nur eine ausgewaehlte Site, weil alle gleichzeitig mehr Ressourcen brauchen.

`wikipedia` braucht zusaetzlich `env setup init --data-dir ...`, weil dort Daten vorbereitet werden. `map` bleibt im aktuellen lokalen Experiment ausgeschlossen, weil Download und Docker-Volumes zu viel Speicher benoetigen.

In [10]:
RUN_PULL_IMAGES = True

if RUN_PULL_IMAGES:
    pull_results = {}
    for site in PREPARE_SITES:
        image = SITE_IMAGES[site]
        print(f'\nPull {site}: {image}')
        result = subprocess.run(['docker', 'pull', image], text=True, capture_output=True)
        pull_results[site] = result.returncode
        print(result.stdout)
        print(result.stderr)
        if result.returncode != 0:
            print(f'FEHLER bei {site}. Du kannst spaeter nur dieses Image erneut ziehen: docker pull {image}')
    pull_results
else:
    print('Image-Download uebersprungen. Setze RUN_PULL_IMAGES = True oder nutze im Terminal:')
    for site in PREPARE_SITES:
        print(f'docker pull {SITE_IMAGES[site]}')
    print('\nExtra Daten-Setup fuer wikipedia:')
    for site in SITES_WITH_EXTRA_DATA:
        print(f'cd {OFFICIAL_REPO} && uv run webarena-verified env setup init --site {site} --data-dir {DATA_DIR}')


Pull shopping: am1n3e/webarena-verified-shopping
Using default tag: latest
latest: Pulling from am1n3e/webarena-verified-shopping
Digest: sha256:3e8cb9b945ea9b1c94ab26dba53e8d12dd0406abbf4bf686fd3bb2b6a5908feb
Status: Image is up to date for am1n3e/webarena-verified-shopping:latest
docker.io/am1n3e/webarena-verified-shopping:latest



Pull shopping_admin: am1n3e/webarena-verified-shopping_admin
Using default tag: latest
latest: Pulling from am1n3e/webarena-verified-shopping_admin
Digest: sha256:d0531dd27ed98d0c459ff9e88118bf2ed8b660b0ed99c38837db46c065a5be13
Status: Image is up to date for am1n3e/webarena-verified-shopping_admin:latest
docker.io/am1n3e/webarena-verified-shopping_admin:latest



Pull reddit: am1n3e/webarena-verified-reddit
Using default tag: latest
latest: Pulling from am1n3e/webarena-verified-reddit
Digest: sha256:0594908059a03e5f610005689440a95c05e776e734006197c7b1eba12734cb46
Status: Image is up to date for am1n3e/webarena-verified-reddit:latest
docker.io/am1n3e/web

KeyboardInterrupt: 

In [11]:
RUN_START_SITE = False

start_cmd = ['uv', 'run', 'webarena-verified', 'env', 'start', '--site', SITE]
if SITE in SITES_WITH_EXTRA_DATA:
    start_cmd += ['--data-dir', str(DATA_DIR)]

if RUN_START_SITE:
    subprocess.run(start_cmd, cwd=OFFICIAL_REPO, check=True)
else:
    print('Start uebersprungen. Setze RUN_START_SITE = True oder starte im Terminal:')
    print(f'cd {OFFICIAL_REPO}')
    print(' '.join(start_cmd))
    print('\nFuer alle Sites nacheinander:')
    for site in ALL_SITES:
        cmd = ['uv', 'run', 'webarena-verified', 'env', 'start', '--site', site]
        if site in SITES_WITH_EXTRA_DATA:
            cmd += ['--data-dir', str(DATA_DIR)]
        print(' '.join(cmd))

Start uebersprungen. Setze RUN_START_SITE = True oder starte im Terminal:
cd /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified
uv run webarena-verified env start --site shopping

Fuer alle Sites nacheinander:
uv run webarena-verified env start --site shopping
uv run webarena-verified env start --site shopping_admin
uv run webarena-verified env start --site reddit
uv run webarena-verified env start --site gitlab
uv run webarena-verified env start --site wikipedia --data-dir /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/downloads


In [12]:
status = subprocess.run(
    ['uv', 'run', 'webarena-verified', 'env', 'status', '--site', SITE],
    cwd=OFFICIAL_REPO,
    text=True,
    capture_output=True,
)
print(status.stdout)
print(status.stderr)
if status.returncode != 0:
    raise RuntimeError(
        f'Die Site {SITE!r} laeuft noch nicht. Setze oben RUN_START_SITE = True '
        f'oder starte im Terminal: cd {OFFICIAL_REPO} && uv run webarena-verified env start --site {SITE}'
    )
print(f'{SITE} ist laut WebArena-Verified env status bereit.')

[WebArena Verified] [INFO] Checking status of container: webarena_verified_shopping
[WebArena Verified] [INFO] Container: webarena_verified_shopping
[WebArena Verified] [INFO] Status: running
[WebArena Verified] [INFO] URL: http://localhost:7770
[WebArena Verified] [INFO] Env-ctrl: http://localhost:7771
[WebArena Verified] [INFO] Services: {
  "success": true,
  "message": "",
  "details": {
    "value": {
      "services": {
        "php-fpm": "RUNNING",
        "nginx": "RUNNING",
        "mailcatcher": "HEALTHY",
        "env-ctrl": "RUNNING",
        "redis-server": "HEALTHY",
        "elasticsearch": "HEALTHY",
        "cron": "RUNNING",
        "mysqld": "HEALTHY"
      }
    },
    "exec_logs": [
      {
        "command": "test -S /run/supervisord.sock",
        "returncode": 0,
        "stdout": "",
        "stderr": ""
      },
      {
        "command": "supervisorctl status",
        "returncode": 3,
        "stdout": "cron                             RUNNING   pid 57, upti

## 3. Lokale Config fuer diese Site schreiben

In [13]:
env_key, base_url, credentials = SITE_ENV[SITE]
env_config = {'urls': [base_url], 'active_url_idx': 0}
if credentials is not None:
    env_config['credentials'] = credentials
if SITE == 'shopping_admin':
    env_config['use_header_login'] = True
config = {'environments': {env_key: env_config}}
config_path = OFFICIAL_REPO / 'output' / f'config.{SITE}.local.json'
config_path.parent.mkdir(exist_ok=True)
config_path.write_text(json.dumps(config, indent=2))
config_path, config

(PosixPath('/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/config.shopping.local.json'),
 {'environments': {'__SHOPPING__': {'urls': ['http://localhost:7770'],
    'active_url_idx': 0,
    'credentials': {'username': 'emma.lopez@gmail.com',
     'password': 'Password.123'}}}})

## 4. Kandidaten-Tasks exportieren

Wir nehmen zuerst `NAVIGATE`-Tasks, weil sie der GitLab-Aufgabe 44 am aehnlichsten sind. Nicht jede Site hat solche Tasks.

In [14]:
candidates_path = OFFICIAL_REPO / 'output' / f'{SITE}_navigate_tasks.json'
proc = subprocess.run([
    'uv', 'run', 'webarena-verified', 'dataset-get',
    '--sites', SITE,
    '--task-type', 'NAVIGATE',
    '--fields', 'task_id,sites,intent,intent_template_id',
    '--output', str(candidates_path.relative_to(OFFICIAL_REPO)),
], cwd=OFFICIAL_REPO, text=True, capture_output=True)
print(proc.stdout)
print(proc.stderr)
if proc.returncode != 0:
    print('Keine NAVIGATE-Tasks fuer diese Site gefunden. Fuer Reddit z. B. spaeter RETRIEVE/MUTATE testen.')
else:
    candidates = json.loads(candidates_path.read_text())
    print('Anzahl Kandidaten:', len(candidates))
    candidates[:10]

[WebArena Verified] [INFO] No config provided, using default configuration
[WebArena Verified] [INFO] Using test_data_file: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json
[WebArena Verified] [INFO] Loading tasks from: '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json'
[WebArena Verified] [INFO] Loaded 812 tasks successfully.
[WebArena Verified] [INFO] WebArenaVerified initialized successfully
[WebArena Verified] [INFO] Wrote 45 tasks to output/shopping_navigate_tasks.json


Anzahl Kandidaten: 45


## 5. Agent-Input fuer einen Kandidaten rendern

In [15]:
TASK_ID = candidates[0]['task_id']
tasks_file = OFFICIAL_REPO / 'output' / f'{SITE}_task_{TASK_ID}.json'
subprocess.run([
    'uv', 'run', 'webarena-verified', 'agent-input-get',
    '--task-ids', str(TASK_ID),
    '--config', str(config_path.relative_to(OFFICIAL_REPO)),
    '--output', str(tasks_file.relative_to(OFFICIAL_REPO)),
], cwd=OFFICIAL_REPO, check=True)

task_input = json.loads(tasks_file.read_text())
task_input

[WebArena Verified] [INFO] Loading config from: '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/config.shopping.local.json'
[WebArena Verified] [INFO] Using test_data_file: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json
[WebArena Verified] [INFO] No config provided, using default configuration
[WebArena Verified] [INFO] Using test_data_file: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json
[WebArena Verified] [INFO] Loading tasks from: '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json'
[WebArena Verified] [INFO] Loaded 812 tasks successfully.
[WebArena Verified] [INFO] WebArenaVerified initialized successfully
[WebArena Verified] [INFO] Wrote 1 agent inputs to output/shopping_task_118.j

[{'sites': ['shopping'],
  'task_id': 118,
  'intent_template_id': 151,
  'start_urls': ['http://localhost:7770'],
  'intent': 'I have a jaw bruxism problem, go to the product page for something that could alleviate the problem.'}]

## 6. BrowserGym Site-Probe

Dieser Probe oeffnet nur die Start-URL mit BrowserGym und schreibt HAR + Metadaten. Er behauptet nicht, die Aufgabe geloest zu haben.

Wenn hier `Connection refused` kommt, ist fast immer die Site-Umgebung nicht gestartet oder noch nicht bereit. Dann zuerst die Status-Zelle oben reparieren.

In [16]:
subprocess.run([
    str(PROJECT_ROOT / '.venv/bin/python'),
    str(PROJECT_ROOT / 'scripts/run_browsergym_site_probe.py'),
    '--tasks-file', str(tasks_file),
    '--task-id', str(TASK_ID),
    '--output-root', str(OFFICIAL_REPO / 'output/site-probe'),
], cwd=PROJECT_ROOT, check=True)

Probe task 118: 100%|██████████| 4/4 [00:17<00:00,  4.32s/step]



Probe artifacts
- metadata: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/site-probe/shopping/118/probe_metadata.json
- network.har: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/site-probe/shopping/118/network.har (4046486 bytes)

Current URL: http://localhost:7770/
Page title: One Stop Market


CompletedProcess(args=['/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/.venv/bin/python', '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/scripts/run_browsergym_site_probe.py', '--tasks-file', '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/shopping_task_118.json', '--task-id', '118', '--output-root', '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/site-probe'], returncode=0)

## 7. Probe-Artefakte ansehen

In [17]:
probe_dir = OFFICIAL_REPO / 'output' / 'site-probe' / SITE / str(TASK_ID)
sorted(p.name for p in probe_dir.iterdir())

['network.har', 'probe_metadata.json']

In [18]:
json.loads((probe_dir / 'probe_metadata.json').read_text())

{'task_id': 118,
 'sites': ['shopping'],
 'intent': 'I have a jaw bruxism problem, go to the product page for something that could alleviate the problem.',
 'start_url': 'http://localhost:7770',
 'current_url_after_reset': 'http://localhost:7770/',
 'page_title_after_reset': 'One Stop Market',
 'observation_keys': ['active_page_index',
  'axtree_object',
  'chat_messages',
  'dom_object',
  'elapsed_time',
  'extra_element_properties',
  'focused_element_bid',
  'goal',
  'goal_object',
  'last_action',
  'last_action_error',
  'open_pages_titles',
  'open_pages_urls',
  'screenshot',
  'url'],
 'runtime_ms': 17316,
 'har_path': '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/site-probe/shopping/118/network.har',
 'note': 'Probe only. This does not solve or evaluate the benchmark task.'}

## Einordnung

Wenn diese Probe funktioniert, ist die naechste Site technisch erreichbar und BrowserGym kann sie oeffnen. Das ist noch keine Benchmark-Loesung. Der naechste echte Schritt waere ein kleiner scripted Agent fuer genau einen einfachen Task auf dieser Site oder danach AgentLab als Experiment-Rahmen.